# ETL — 12 quyển / 3 NXB / 2 399 trang · cập nhật 2026-08-28 (lượt 4, D-145/D-146)

> ## 🆕 Lượt 4 (2026-08-28): Bước 2+3 Hybrid Tesseract+MinerU cho công thức Hoá/Lý
>
> Notebook này nay bật `FORMULA_HYBRID_ENABLED=true` + `TEXT_EXTRACTION_VERSION=
> v3_formula_hybrid` (mục 5a), cài `mineru_vl_utils` ghim `transformers>=4.49,<5`
> (mục 5c), và **DB chuyển sang session-local `/content/database`** (Drive đầy) —
> vòng lặp ETL §9 chạy TỪNG QUYỂN, zip + tải về SAU MỖI QUYỂN để rớt phiên chỉ mất
> tối đa 1 quyển. `min_sat` per-book (`MIN_SAT_FLOOR=9`, D-146) và
> `single_line_max_h` per-book (D-145 Task 8, hiện vẫn dùng mặc định 60px cho cả
> 12 quyển vì mẫu dòng đơn đo được < 5) đã nối sẵn trong code, không cần chỉnh gì
> thêm ở notebook cho hai việc đó.
>
> ## ⚠️ `--build-manifests` NAY CHẠY ĐƯỢC CẢ 12 QUYỂN — nhưng spine Bài chưa nghiệm thu
>
> Cập nhật 2026-08-23 (D-70): `book/toc_lines.py` thêm bộ đọc MỤC LỤC **theo dòng**
> cho CTST + Cánh Diều, và `read_toc` chọn bộ đọc theo **fingerprint đã đo**. Nên
> `--build-manifests` **không còn bị chặn** — chạy hết 12 quyển, đo được
> **1,27–1,46 s/trang** → ~50 phút cho 2 399 trang.
>
> **Nhưng chưa nghiệm thu:** spine Bài của **8 quyển CTST/CD chưa liền mạch**
> (đọc được CTST 23/17/17/21 và CD 32/24/23/29 mục; gần nhất là 6_CD, thiếu Bài
> 3, 4, 34). Hệ quả: **cổng G1 vẫn FAIL cho 8 quyển đó**, và `bai_so` **không**
> được ghi vào metadata chunk — đúng thiết kế: thiếu thì im, không đoán.
>
> **Điều đó KHÔNG chặn text ETL:** `save_manifest` ghi manifest cho từng quyển
> dựng xong bất kể G1, và `--text-only` chỉ cần manifest tồn tại. Đừng nối hai
> lệnh bằng `&&` — mã thoát 1 của G1 sẽ chặn bước 2 một cách vô lý.

**Nguồn:** `datasources/` là **12 thư mục PNG, một file mỗi trang**
(`SGK_KHTN_{6,7,8,9}_{KNTT,CTST,CD}/page_001.png …`), **2 399 trang**, 0 khoảng
trống, mọi quyển bắt đầu từ `page_001`.

```
CD   179 + 171 + 207 + 215 = 772 | CTST 204 + 188 + 223 + 215 = 830
KNTT 195 + 179 + 196 + 227 = 797                        tổng 2 399
```

**Năm thứ ĐÃ ĐỔI so với lượt 2 — mỗi cái làm hỏng một ô của notebook cũ:**

1. **`datasources/` KHÔNG còn trong git (D-68).** Bản clone chỉ có
   `datasources/README.md`. **Phải trỏ `RAG_DATA_DIR` sang Drive** (mục 5). Trỏ
   sai chỗ thì cả bốn entrypoint **thoát mã 2** kèm thông báo — không còn cảnh
   "chạy xong mà không xử lý gì".
2. **`--build-manifests` nay BẮT BUỘC** (ngược hẳn lượt 2). `database/manifests/`
   đã bị xoá cùng corpus cũ; đường text **raise `ManifestMissing`** nếu thiếu.
3. **`offset = 0`, không còn `−1` (D-65).** `printed_page == số trong tên file`,
   đo 40 trang/quyển trên 12/12 quyển. `page_001` **không** còn là bìa mặc định.
4. **KNTT là bộ ĐỘ PHÂN GIẢI THẤP NHẤT**, không phải bộ tham chiếu: KNTT
   1094×1536 vs CTST/CD **2280×3201** (6_CD 2480×3480) — ~3,5 lần diện tích. Nên
   **mọi con số s/trang trong notebook cũ chỉ đúng cho KNTT** và là **cận dưới**
   cho CD/CTST. Đừng hứa lịch theo chúng; đọc `s/trang` mà chính log ETL in ra.
5. **`RAG_FINGERPRINT_DIR` là biến MỚI** (mục 5). Fingerprint layout M0
   (`database/fingerprints/*.json`, 12/12 quyển) đã commit trong repo và đi theo
   **repo**, không theo Drive — cùng lý do như manifest: nó là kết quả đo, một
   lượt đo lại tốn ~70 phút OCR.

**Vẫn đúng từ lượt 2, đừng đổi:**

- **Caption ảnh TẮT** (`IMAGE_CAPTION_ENABLED=false`, D-47) — đo trên 12 crop
  thật: 17,6 s/crop trên CPU, **4/12 caption bịa** chi tiết không có trong ảnh,
  **0/4** lần tự nêu số hiệu hình là đúng. Bật lại mà không đo = tự bắn vào chân.
- **Checkpoint khoá theo hash TỪNG TRANG + version.** Sửa 1 trang → chỉ trang đó
  chạy lại; **bump version là cách DUY NHẤT** ép làm lại toàn bộ một phía.
- Text embedding **`BAAI/bge-m3`** (1024 chiều) + cross-encoder
  **`BAAI/bge-reranker-v2-m3`**. Đổi model embedding thì **phải dựng lại index**.
- Secret lấy từ **Colab Secrets** (🔑), không hardcode.

**Chi phí — ngoại suy, PHẢI đo lại trên một quyển CD trước khi hứa lịch:** text
3,56 s/trang × 2 399 ≈ **2,4 giờ**; ảnh 8,86 s/trang × 2 399 ≈ **5,9 giờ**. Cả hai
đo trên KNTT 1094×1536 nên với CD/CTST (3,5 lần diện tích) chúng là **cận dưới**.

> ⚠️ **Bảo mật:** notebook này từng hardcode `HF_TOKEN` trong ô mã (đã lộ vào git
> history). **Revoke token HF cũ + GitHub PAT cũ**, tạo token mới, lưu vào Colab
> Secrets tên `HF_TOKEN`.


## 1. Clone repo

In [ ]:
# Pipeline nguồn PNG đã merge vào master.
!git clone -b master https://github.com/lcdkhoa/project-bio-rag.git

In [ ]:
%cd project-bio-rag
!git log --oneline -3

## 2. Cài dependencies + Tesseract (vie)

`poppler-utils` chỉ còn cần cho đường upload PDF legacy (`/api/etl`) — nguồn PNG
không dùng. Cài luôn cho chắc, nhẹ.

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-vie
!tesseract --version | head -1 && tesseract --list-langs | grep -x vie

## 3. Secret + env cơ bản (đặt TRƯỚC khi tải model)

`HF_TOKEN` lấy từ Colab Secrets. Mở tab 🔑 (Secrets) bên trái, thêm khoá
`HF_TOKEN`, bật *Notebook access*.

In [ ]:
import os, multiprocessing
from google.colab import userdata

# HF token từ Colab Secrets — KHÔNG hardcode (bản trước hardcode và đã làm lộ token).
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Đa luồng khớp số CPU thực tế
n = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(n)
os.environ["NUMEXPR_NUM_THREADS"] = str(n)
os.environ["OPENBLAS_NUM_THREADS"] = str(n)

os.environ["USE_GPU"] = "true"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| CPU cores:", n)

## 4. Tải model về `./models` (chạy ONLINE)

Tải **cả 6 model là ~15 GB**. Chỉ tải thứ mình cần:

| profile | model | dùng cho |
|---|---|---|
| `text-etl` | bge-m3 (~2 GB) | `--text-only` |
| `image-etl` | CLIP + OWL-ViT + Vintern (~9 GB) | `--image-only` |
| `serve` | bge-m3 + reranker + Qwen-3B + CLIP (~12 GB) | `--api` |
| `all` | tất cả (~15 GB) | mặc định |

> **Caption đã tắt (D-47) nên `--image-only` KHÔNG cần Vintern-1B (~2 GB).** Profile
> `image-etl` vẫn tải nó để giữ đúng nghĩa "đủ cho phía ảnh nếu bật caption"; muốn
> nhẹ thì tải đúng hai model cần:
> `--only clip-vit-base-patch16,owlvit-base-patch32`.

> **`serve` bắt buộc có `bge-reranker-v2-m3`.** Thiếu nó thì với `HF_HUB_OFFLINE=1`,
> `RerankedRetriever` chỉ log một `warning` MỖI TRUY VẤN rồi rơi về xếp theo khoảng
> cách — `RERANK_ENABLED=true` mà thực chất không rerank. Cổng G3 in ra dòng
> `rerank: ...` để bạn thấy chuyện đó; đừng bỏ qua nó.

Trên **Colab free** hãy dùng `--profile text-etl` cho lượt đầu: nhanh hơn nhiều và
không ăn hết disk. `HF_HUB_OFFLINE` chưa bật ở bước này để tải được.

In [ ]:
# Luot ETL text: chi can bge-m3
!python ./src/utils/download_models.py --save_dir ./models --profile text-etl

# Phia anh - caption da tat nen chi can detector + CLIP (nhe hon profile image-etl):
# !python ./src/utils/download_models.py --save_dir ./models --only clip-vit-base-patch16,owlvit-base-patch32

# Serve API (BAT BUOC co reranker, xem ghi chu o tren):
# !python ./src/utils/download_models.py --save_dir ./models --profile serve

## 5. Mount Drive + trỏ DB ra Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# Index nặng (ChromaDB + ảnh crop) -> Drive, bền qua các phiên Colab.
# D-56 Bước 2/3 (2026-08-28): Drive đầy — DB nay ở SESSION, KHÔNG Drive.
# RỦI RO: mất checkpoint-resume nếu Colab rớt phiên (khác bản Drive cũ "bền
# qua các phiên"). Giảm thiểu: chạy TỪNG QUYỂN MỘT (mục 9b), tải zip về SAU
# MỖI QUYỂN — rớt phiên giữa chừng chỉ mất tối đa 1 quyển.
os.environ["RAG_DATABASE_DIR"] = "/content/database"
os.environ["FORMULA_HYBRID_ENABLED"] = "true"
os.environ["TEXT_EXTRACTION_VERSION"] = "v3_formula_hybrid"

# ĐỔI TỪ 2026-08-23 (D-68): `datasources/` KHÔNG còn trong git — bản clone chỉ có
# README.md. Ảnh trang PHẢI nằm trên Drive (hoặc copy về /content). Cấu trúc:
#   <RAG_DATA_DIR>/SGK_KHTN_6_KNTT/page_001.png ...
# Trỏ vào chỗ trống thì main.py thoát mã 2, không âm thầm chạy 0 quyển.
os.environ["RAG_DATA_DIR"] = "/content/drive/MyDrive/project_bio_rag/datasources"

# Manifest (bản đồ trang + spine Bài) đi theo REPO, không theo Drive.
os.environ["RAG_MANIFEST_DIR"] = "/content/project-bio-rag/database/manifests"

# MỚI: fingerprint layout M0 (12/12 quyển) cũng đi theo REPO — nó là KẾT QUẢ ĐO đã
# commit, đo lại tốn ~70 phút OCR. Không đặt biến này thì mặc định đã là
# <repo>/database/fingerprints (đường dẫn tuyệt đối, không phụ thuộc cwd — D-69).
os.environ["RAG_FINGERPRINT_DIR"] = "/content/project-bio-rag/database/fingerprints"

for key in ("RAG_DATABASE_DIR", "RAG_DATA_DIR", "RAG_MANIFEST_DIR",
            "RAG_FINGERPRINT_DIR", "FORMULA_HYBRID_ENABLED", "TEXT_EXTRACTION_VERSION"):
    print(f"{key} = {os.environ[key]}")

# Đọc ảnh 2280x3201 trực tiếp từ Drive rất chậm. Nếu chạy cả CD/CTST, copy về đĩa
# local của Colab trước rồi trỏ lại (bỏ comment):
# !mkdir -p /content/datasources && cp -r "$RAG_DATA_DIR"/* /content/datasources/
# os.environ["RAG_DATA_DIR"] = "/content/datasources"



### 5c. Cài MinerU cho hybrid công thức (D-56 Bước 2/3)

Ghim `transformers>=4.49,<5` — D-101 đo được 5.x nạp HỎNG lm_head của model họ
Qwen2-VL (MinerU dùng kiến trúc này), sinh token rác thay vì đọc kém.

In [ ]:
!pip install -q mineru_vl_utils "transformers>=4.49,<5"
import transformers
print("transformers:", transformers.__version__)
assert transformers.__version__.split(".")[0] != "5" or \
    int(transformers.__version__.split(".")[1]) == 0, \
    "Ghim sai — kiểm lại 'transformers>=4.49,<5' có hiệu lực chưa"

### 5b. Kiểm tra nguồn trước khi chạy bất cứ thứ gì

Kỳ vọng (đo 2026-08-23, D-65) — **12 quyển, 2 399 trang, 0 khoảng trống**:

| NXB | số trang 6/7/8/9 | kích thước |
|---|---|---|
| KNTT | 195 / 179 / 196 / 227 = 797 | 1094×1536 (RGBA) |
| CTST | 204 / 188 / 223 / 215 = 830 | 2280×3201 (RGBA) |
| CD | 179 / 171 / 207 / 215 = 772 | 2280×3201; **6_CD là 2480×3480** (RGB) |

Ô dưới in `thieu: []` cho **mọi** quyển thì mới chạy tiếp. Thiếu quyển nào tức
`RAG_DATA_DIR` chưa trỏ đúng chỗ trên Drive — sửa mục 5, đừng chạy tiếp.


In [ ]:
import torch
from src.etl.page_source import discover_page_sources
import os

print("CUDA:", torch.cuda.is_available())
total = 0
for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
    numbers = source.page_numbers()
    gaps = [n for n in range(numbers[0], numbers[-1] + 1) if n not in set(numbers)]
    total += len(numbers)
    print(f"{source.name}: {len(numbers)} trang, {numbers[0]}..{numbers[-1]}, thieu: {gaps}")
print("TONG:", total, "| shape trang mau:", source.load(numbers[0]).shape)

> **(Tuỳ chọn) Build SẠCH từ đầu.** Checkpoint resume sẽ *bỏ qua* trang đã xử lý.
> Muốn dựng lại hoàn toàn, bỏ comment ô dưới (⚠️ **XOÁ toàn bộ index ở
> `RAG_DATABASE_DIR`**). Manifest KHÔNG bị xoá vì nó nằm trong repo
> (`RAG_MANIFEST_DIR`) — đó là điều mong muốn: bản đồ trang không cần dựng lại.
>
> Cách nhẹ hơn mà không xoá gì: **bump `TEXT_EXTRACTION_VERSION`** (mục 6) — mỗi
> trang sẽ được OCR lại và chunk cũ của chính trang đó bị xoá trước khi ghi mới.

In [ ]:
# import shutil, os
# shutil.rmtree(os.environ["RAG_DATABASE_DIR"], ignore_errors=True)
# os.makedirs(os.environ["RAG_DATABASE_DIR"], exist_ok=True)
# print("Đã xoá sạch:", os.environ["RAG_DATABASE_DIR"])

## 6. Env runtime — trỏ model local + bật offline + version gate

`HF_HUB_OFFLINE=1` (model đã tải ở bước 4). Hai biến version ở cuối ô là **cách
duy nhất** ép làm lại toàn bộ một phía; không đổi thì lượt chạy sau skip sạch (đó
là ý muốn, không phải lỗi).

In [ ]:
import os
base = "/content/project-bio-rag/models"

os.environ["HF_HUB_OFFLINE"] = "1"          # model da tai o buoc 4

# --- Text: bge-m3 + reranker ---
os.environ["EMBEDDING_MODEL"] = f"{base}/bge-m3"
os.environ["RERANK_ENABLED"] = "true"
os.environ["RERANK_MODEL"]  = f"{base}/bge-reranker-v2-m3"

# --- LLM + image models ---
os.environ["LLM_MODEL"]     = f"{base}/Qwen2.5-3B-Instruct"
os.environ["CLIP_MODEL"]    = f"{base}/clip-vit-base-patch16"
os.environ["OWL_VIT_MODEL"] = f"{base}/owlvit-base-patch32"

# --- Caption anh: TAT (D-47) ---
# Da do tren 12 crop that: 17,6 s/crop tren CPU, 4/12 caption BIA chi tiet khong
# co trong anh, 0/4 lan model tu neu so hieu hinh la dung. Dung bat lai ma khong
# do lai. Bat ma model khong nap duoc thi ETL RAISE, khong nuot loi.
os.environ["IMAGE_CAPTION_MODEL"]   = f"{base}/Vintern-1B-v2"   # tro khi tat
os.environ["IMAGE_CAPTION_ENABLED"] = "false"

# --- Version gate: DOI GIA TRI = ep lam lai toan bo phia do ---
# Ca hai deu da bump so voi luot chay truoc, nen neu Drive con index cu thi ca
# hai phia se chay lai tu dau - dung y muon, vi ket qua hai phia deu da doi.
os.environ["TEXT_EXTRACTION_VERSION"]  = "v2_bai_spine"       # spine Bai -> bai_so vao metadata
os.environ["IMAGE_EXTRACTION_VERSION"] = "v19_pill_kernels"   # pill hop nhieu kernel CLOSE
print("text ver :", os.environ["TEXT_EXTRACTION_VERSION"])
print("image ver:", os.environ["IMAGE_EXTRACTION_VERSION"])
print("caption  :", os.environ["IMAGE_CAPTION_ENABLED"], "(tat = dung, xem D-47)")

## 7. BƯỚC 0 — `BookManifest` + cổng G1  ·  **BẮT BUỘC, ĐỪNG BỎ QUA**

Manifest là **nguồn sự thật duy nhất về số trang in**; đường text dừng với
`ManifestMissing` nếu thiếu, nó không đoán.

> ### Ngược hẳn lượt 2 của notebook này
> Lượt 2 ghi "manifest đã commit trong repo → bỏ qua ô này". Câu đó **hết hiệu
> lực**: `database/manifests/` đã bị xoá cùng corpus 4 quyển cũ, nên **không có
> manifest nào trong repo**. Phải chạy ô dưới, và **commit manifest mới** sau khi
> G1 PASS.

**Chạy được / nghiệm thu được, nói theo phép đo (D-65 + D-70):**

| NXB | `--build-manifests` | spine Bài | ghi chú |
|---|---|---|---|
| KNTT (4 quyển) | **chạy được** | **LIỀN MẠCH 195/195**, G1 PASS | bộ đọc BẢNG (`toc.read_toc_cells`): MỤC LỤC đầu sách `[4,5]`, mục `Bài N`, cột số trang riêng |
| CTST (4 quyển) | **chạy được** (D-70) | **chưa liền mạch**: 23/17/17/21 mục | bộ đọc THEO DÒNG (`toc_lines`): `BÀI N:` chữ hoa, hai cột (khe 99–123 px, tâm x≈0,50 ở 8/8 trang), dot leader dày làm số trang OCR ra rác → ~20 cờ `toc_page_unreadable`/quyển |
| Cánh Diều (4 quyển) | **chạy được** (D-70) | **chưa liền mạch**: 32/24/23/29 mục | mục `N. Tiêu đề` (không có chữ "Bài"), MỤC LỤC ở **hai trang CUỐI**; 6_CD/7_CD hai cột nhưng **8_CD/9_CD MỘT cột** chạy hết bề rộng |

Chạy hết 12 quyển bằng một lệnh (mã thoát 1 = G1 FAIL, **dự kiến** cho 8 quyển
CTST/CD; manifest vẫn được ghi):

    python main.py --build-manifests

Hoặc một quyển:

    python main.py --build-manifests --book SGK_KHTN_6_KNTT

**Cái bẫy đắt nhất, đã đo trên 7_CTST:** chạy bộ đọc MỤC LỤC một cột lên bố cục
hai cột sinh ra **số SAI MÀ TRÔNG HỢP LÝ** (`Bài 1 → trang 144`, thật là trang 6),
rồi ràng buộc đơn điệu giết 31 Bài còn lại. Luật "bỏ entry chứ không đoán" chặn 31
số sai nhưng **không chặn số sai đầu tiên**. Vì vậy bộ đọc cell chỉ chạy khi
`entry_style == "bai"`, và spine không liền mạch bị gắn cờ `KHONG_dang_tin`.

Ba điều cần hiểu đúng khi đọc báo cáo G1 (vẫn đúng từ lượt 2):

- **MỤC LỤC dựng spine, huy hiệu Bài chỉ XÁC NHẬN** và không bao giờ ghi đè (D-44
  đảo ngược luật cũ "banner thắng"). Lệch thì ghi `banner_toc_mismatch`.
- **`0/k` ở huy hiệu của sách 7/8/9 là con số THẬT, không phải lỗi cấu hình** —
  lục giác màu đặc chữ trắng, đã thử ba cách đọc, vẫn chưa đọc được. Nó được in ra
  chứ không bị che.
- `bai_so` chỉ đi vào metadata chunk khi spine sạch; quyển bị cờ
  `bai_numbers_not_contiguous`/`spine_out_of_order` thì tự động thôi ghi.


In [ ]:
!python main.py --build-manifests

In [ ]:
# Xem nhanh manifest đã dựng (KHÔNG chạy lại OCR)
import glob, json, os

for path in sorted(glob.glob(os.path.join(os.environ["RAG_MANIFEST_DIR"], "*.json"))):
    m = json.load(open(path, encoding="utf-8"))
    covers = [p["page_index"] for p in m["pages"] if p["role"] == "cover"]
    unread = [p["page_index"] for p in m["pages"]
              if p["source"] != "ocr_confirmed" and p["role"] != "cover"]
    kinds = {}
    for flag in m["flags"]:
        kinds[flag["kind"]] = kinds.get(flag["kind"], 0) + 1
    print(f"{m['book_id']}: {m['n_pages']} trang | offset {m['page_offset']} | "
          f"bìa {covers} | trang có số mà KHÔNG đọc được: {unread} | Bài {len(m['bai'])}")
    print("   flags:", kinds)

## 8. ETL — TEXT (dùng được)

OCR theo **vùng layout** (không phải cả trang) → chunk → ChromaDB. Trang bìa
(`role="cover"`) bị bỏ qua ở bước chunk, **file nguồn không bị xoá**.

Resume theo TỪNG TRANG: Colab ngắt giữa đường thì chạy lại đúng lệnh này, nó chỉ
làm phần còn thiếu và **không nhân bản chunk**.

~1,6 s/trang → **~21 phút cho 801 trang** trên 1 luồng CPU (OCR không dùng GPU).

In [ ]:
# Ước lượng chi phí / số dòng gọi MinerU trước khi chạy ETL thật
!python -m src.test.estimate_formula_hybrid_cost


In [ ]:
import subprocess

BOOKS = ["SGK_KHTN_6_KNTT", "SGK_KHTN_7_KNTT", "SGK_KHTN_8_KNTT", "SGK_KHTN_9_KNTT",
         "SGK_KHTN_6_CTST", "SGK_KHTN_7_CTST", "SGK_KHTN_8_CTST", "SGK_KHTN_9_CTST",
         "SGK_KHTN_6_CD", "SGK_KHTN_7_CD", "SGK_KHTN_8_CD", "SGK_KHTN_9_CD"]

for book in BOOKS:
    print(f"\n=== {book} ===")
    r = subprocess.run(["python", "main.py", "--text-only", "--book", book])
    print(f"{book} exit code: {r.returncode}")
    if r.returncode != 0:
        print(f"!! {book} THẤT BẠI (mã {r.returncode}) — DỪNG, kiểm log trước "
              "khi sang quyển tiếp theo (đừng chạy tiếp che lấp lỗi).")
        break
    !zip -r -q /content/database_backup.zip /content/database
    from google.colab import files
    files.download('/content/database_backup.zip')
    print(f"Đã tải zip sau khi xong {book} (zip CỘNG DỒN các quyển trước đó).")


In [ ]:
# Còn bao nhiêu trang chưa index? (biết có bị ngắt giữa đường không)
# Chạy trong subprocess để không giữ model embedding trong RAM của notebook.
import subprocess, sys, textwrap

script = textwrap.dedent("""
    import os
    from src.etl import ProcessingStatus
    from src.etl.page_source import discover_page_sources

    status = ProcessingStatus()
    for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
        print(source.name,
              "| text còn thiếu:", len(status.pages_needing_text(source)),
              "| ảnh còn thiếu:", len(status.pages_needing_images(source)))
""")
done = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
print(done.stdout or done.stderr[-2000:])

## 8b. Chỉ mục THƯA (BM25) — bắt buộc nếu dùng truy xuất lai

Đề cương Nội dung 2 đòi **truy xuất lai BM25 + dense**. Chỉ mục thưa dựng **từ chính `biology_text`**, nên:

* **KHÔNG OCR lại gì cả** — đo được **5,5 s** cho 16 393 chunk (so với `--text-only` là 3 giờ 20).
* Phải chạy **SAU** `--text-only`, và **chạy lại mỗi khi index text đổi**.

Không phải nhớ điều đó bằng kỷ luật: chỉ mục thưa mang **dấu vân của index** (số chunk + digest md5 của tập `chunk_id` + `TEXT_EXTRACTION_VERSION` + tokenizer + phiên bản bộ chuẩn hoá). Lệch bất kỳ trường nào là `SparseIndexStale` **ném lỗi ra**, không âm thầm dùng bản cũ — một chỉ mục thưa cũ hơn index trả về `chunk_id` không còn tồn tại, và cách hỏng đó im lặng (cùng loại D-52).


In [ ]:
!python main.py --build-bm25

# Đầu ra mong đợi (đo trên index 12 quyển, 2026-08-24):
#   [bm25] 16393 chunk, 19727 từ vựng, độ dài TB 56.8 token
#   [bm25] dấu vân: n_chunks=16393 ids_digest=… text_version=v2_bai_spine
#          tokenizer=plain normalizer=v3_formula_skeleton_plus_letters


**Bật truy xuất lai** — mặc định `RETRIEVAL_MODE` vẫn là `dense` (= hành vi cũ, có chủ ý: chỉ đổi mặc định khi bảng 12 cấu hình có số). Ba chế độ:

| `RETRIEVAL_MODE` | nghĩa |
|---|---|
| `dense` | chỉ vector (bge-m3) — **mặc định** |
| `bm25` | chỉ từ khoá |
| `hybrid` | hợp nhất cả hai, rồi cổng lọc, rồi rerank |

`RELEVANCE_GATE_ENABLED` nay **tách rời** `RERANK_ENABLED` (trước M2 hai thứ này loại trừ nhau, nên `RETRIEVER_DISTANCE_MARGIN` là số chết — D-80).


In [ ]:
import os

# Mặc định ĐÃ CHỐT BẰNG SỐ trên 300 câu / 12 quyển ở ĐÚNG bề rộng production
# (20 ứng viên/kênh) — D-82. Đừng đổi bằng cảm tính.
os.environ['RETRIEVAL_MODE'] = 'hybrid'         # dense | bm25 | hybrid
# CỔNG LỌC TƯƠNG ĐỐI: TẮT. Bản notebook trước ghi 'true' — SAI. Đo được nó
# CÓ HẠI ở bề rộng thật: hybrid MRR 0,808 -> 0,781 và R@10 0,977 -> 0,930.
# Cổng lọc thực sự hoạt động là sàn tuyệt đối RERANK_SCORE_MIN.
os.environ['RELEVANCE_GATE_ENABLED'] = 'false'
os.environ['RERANK_ENABLED'] = 'true'           # cross-encoder
# Đã chọn BẰNG PHÉP QUÉT (lưới 5×5 trên 100 câu), đừng đổi bằng cảm tính:
os.environ['BM25_K1'] = '0.7'
os.environ['BM25_B'] = '0.75'
os.environ['BM25_TOKENIZER'] = 'plain'   # GIỮ dấu — đo được thắng BỎ dấu
# Ngữ cảnh đa phương thức (Mục tiêu 4, cấu hình 2 — D-85). Mặc định 'false' =
# hành vi cũ. Bật lên thì nhãn + chú thích hình DETERMINISTIC (pill/OCR, KHÔNG
# do model sinh) được nối vào ngữ cảnh LLM.
os.environ['MULTIMODAL_CONTEXT_ENABLED'] = 'false'
print({k: v for k, v in os.environ.items()
       if k.startswith(('RETRIEVAL_', 'BM25_', 'RERANK_', 'RELEVANCE_',
                        'MULTIMODAL_'))})


## 9. ETL — ẢNH: chỉ chạy **4 quyển KNTT**. Đã chạy thật 2026-08-25, có số.

Crop figure + index CLIP. `IMAGE_EXTRACTION_VERSION=v19_pill_kernels`.

**Kết quả lượt chạy thật (máy dev 16 lõi, KHÔNG GPU) — dùng số này, đừng dùng số
của corpus 801 trang đã bị xoá:**

| quyển | trang | hình | s/trang (pha crop) |
|---|---|---|---|
| 6_KNTT | 195 | 285 | 9,10 |
| 7_KNTT | 179 | 203 | 11,11 |
| 8_KNTT | 196 | 215 | 6,75 |
| 9_KNTT | 227 | 235 | 6,51 |
| **tổng** | **797** | **938** | **20:40 → 23:04 = 2 h 24** |

Biên độ giữa các quyển gần **1,7×**, nên đừng suy một con số từ một quyển. OCR neo
caption chạy trước mỗi quyển, riêng nó **0,79 s/trang**.

Kho sau khi dựng: `figure_label` **891/938**, `crop_text` 806, `figure_caption`
557, `visual_caption_vi` **0/938** (captioner TẮT, đúng D-47), **578/797 trang có
hình = 72,5%**.

**Cổng G4 chạy lại trên corpus mới** (`--bai-per-book 4`, cả 4 quyển): **gán SAI
Bài 0/0/0/0, thiếu (cận dưới) 0/0/0/0**, 72 hình có nhãn / 12 không nhãn.
Cột đáng xem: **`crop nghi cắt lấn` 30/86 = 34,9%** — đó là **CỜ cho người**, phần
lớn là `activity_box` (hộp hoạt động vốn nhiều chữ). Ca cần mở ra xem:
`9_KNTT` trang 12 `Hình 1.12`, diện tích **0,749** trang.

**Hai điều KHÔNG được làm:**

1. **Đừng chạy `--image-only` không tham số** — nó chạy cả 12 quyển ≈ 6 giờ, trong
   đó 8 quyển CD/CTST đã đo được là sai: kênh pill đọc **0 nhãn** trên cả 8 (D-65),
   vì CD/CTST dùng caption chữ đen còn KNTT dùng pill. `--book` **chỉ lọc được từ
   D-84**; trước đó cờ này bị bỏ qua âm thầm ở đường `--image-only`.
2. **Đừng bật `IMAGE_CAPTION_ENABLED`** (Vintern-1B bịa 4/12 crop — D-47).

**Nhớ:** doc ảnh khoá theo hash của **crop** (D-52), nên lượt chạy **thứ hai** phải
để `delete_page_documents` xoá doc cũ, không thì còn doc mồ côi.

**Số "thiếu = 0" của G4 vẫn chỉ là CẬN DƯỚI:** hình cuối của một Bài mất thì
`max(B)` tụt theo và không ai biết — đã xảy ra thật với `Hình 2.5` sách 8.


In [ ]:
# ĐÚNG 4 quyển KNTT, mỗi quyển một lượt (xem markdown ở trên vì sao KHÔNG
# chạy cả 12 quyển). Tên quyển sai -> thoát mã 2 chứ không âm thầm xử lý 0 quyển.
!python main.py --image-only --book SGK_KHTN_6_KNTT
!python main.py --image-only --book SGK_KHTN_7_KNTT
!python main.py --image-only --book SGK_KHTN_8_KNTT
!python main.py --image-only --book SGK_KHTN_9_KNTT

# Rồi CHẠY LẠI CỔNG và đọc số của chính lượt này (đừng trích số cũ):
!python -m src.test.qa_figures --book SGK_KHTN_6_KNTT --bai-per-book 4
!python -m src.test.qa_figures --book SGK_KHTN_7_KNTT --bai-per-book 4
!python -m src.test.qa_figures --book SGK_KHTN_8_KNTT --bai-per-book 4
!python -m src.test.qa_figures --book SGK_KHTN_9_KNTT --bai-per-book 4

# Bảng cấu hình 2 của Giai đoạn 3 (text-only vs multi-modal), 0 lượt gọi LLM.
# Đo ~21 s/câu -> 100 câu KNTT ~35 phút.
!python -m src.test.ablation_multimodal --bo-sach KNTT


In [ ]:
# Vòng review người cho metadata ảnh (ngữ nghĩa file JSON rất dễ hiểu sai —
# đọc README §6 trước khi dùng). Xoá dấu # để chạy.
# !python main.py --export-image-review database/review_images.json
# ... sửa file JSON ...
# !python main.py --apply-image-review database/review_images.json --review-user khoa

### 9b. (Thay thế) `--etl` = text + ảnh trong một lượt

Chỉ dùng khi bạn chấp nhận trạng thái của phía ảnh ở mục 9. Đường text bên trong
`--etl` **giống hệt** `--text-only`.

In [ ]:
# !python main.py --etl

### 9c. Lượt chạy thử THẬT đã kiểm chứng notebook này (2026-08-21)

Không phải mô tả kỳ vọng — đây là log của một lượt chạy thật trên **máy dev
(Windows, 16 core, KHÔNG CUDA)** với **corpus scratch 12 trang** và **DB riêng**,
đúng đường `main.py --etl` mà notebook này gọi:

```
RAG_DATA_DIR=<scratch>/scratch_corpus        # SGK_KHTN_6_KNTT/page_020..031.png (12 trang)
RAG_DATABASE_DIR=<scratch>/db_colab          # DB riêng, không đụng database/ của repo
RAG_MANIFEST_DIR=<repo>/database/manifests   # manifest ĐÃ COMMIT, không chạy --build-manifests
```

**Lượt 1 — chạy hết, không cần can thiệp gì (3 phút 24 giây):**

```
[SGK_KHTN_6_KNTT] Extracting images: 100%|##########| 12/12 [02:02<00:00, 10.22s/it]
[SGK_KHTN_6_KNTT] Extracted 17 images from 12 pages
Added 17 image metadata docs to ImageVectorDB
Added 17 visual image docs to ImageVectorDB
Completed: SGK_KHTN_6_KNTT
ETL (FULL) pipeline completed!
```

Không có dòng `caption model unavailable` — vì caption **tắt tường minh**, không
phải tắt âm thầm như trước (D-42 → D-47). `pill` đọc được nhãn hình trên đường đi,
ví dụ `[pill] 3 nhãn hình đọc được từ pill: ['Hình 10.3', 'Hình 10.1', 'Hình 10.2']`.

**Lượt 2 — chạy lại ĐÚNG lệnh đó: bỏ qua sạch (12 giây, gần hết là nạp model):**

```
VectorDB initialized with 76 existing chunks
Previously processed files: text=1, images=1
[SGK_KHTN_6_KNTT] Already processed for both text and images, skipping
ETL (FULL) pipeline completed!
```

Đây là bằng chứng cho câu "Colab ngắt giữa đường thì chạy lại đúng lệnh này".

**Lượt 3 — checkpoint khoá theo HASH NỘI DUNG trang, không theo tên file.** Ghi nội
dung của `page_040.png` lên `page_025.png` (md5 đổi `fbe5f29c…` → `8dff0f99…`) rồi
chạy lại:

```
[SGK_KHTN_6_KNTT] Extracted 2 images from 1 pages
[SGK_KHTN_6_KNTT] xoá 1 doc ảnh cũ của 1 trang trước khi ghi bản mới
[SGK_KHTN_6_KNTT] Added 2 images to ImageVectorDB
```

**Đúng 1 trang** chạy lại, 11 trang còn lại không bị đụng. Kiểm tra DB sau đó:
`text còn thiếu: []`, `ảnh còn thiếu: []`, và chunk text đúng một mục cho mỗi
`page_index` 20..31 — **không mồ côi, không nhân bản**.

> 🐞 **Lượt chạy thử này tìm ra một lỗi thật (D-52), và nó đã được sửa ngay.** Trước
> khi sửa, lượt 3 làm trang 25 có **3** doc ảnh (`Hình 8.1` cũ + `Hình 11.6`/`Hình
> 11.7` mới) vì đường ảnh **không xoá** doc cũ: `image_id` là hash của CROP nên crop
> đổi thì id đổi và doc cũ không bị upsert đè. Crop mồ côi vẫn tra ra được — học
> sinh có thể được trả về một hình không còn tồn tại trên trang. Chuyện này **sẽ
> xảy ra với MỌI trang** ở lần bump `IMAGE_EXTRACTION_VERSION` này nếu không sửa.
> Nay `ImageVectorDB.delete_page_documents` xoá đúng những trang sắp ghi lại, trên
> cả hai collection ảnh. Đó là lý do phải CHẠY THỬ chứ không chỉ đọc code.

## 10. (Tuỳ chọn) Eval — Recall@k / MRR / **cổng G3**

Ba nhóm script, hai loại:

**Không cần LLM** (chạy được ngay):

    python -m src.test.qa_citation_page      # cổng G3: trang được TRÍCH DẪN có chứa câu trả lời?
    python src/test/recall_at_k.py           # recall@k + MRR, base vs rerank

**Cần `EVAL_LLM_*`** (endpoint OpenAI-compatible bất kỳ):

    python src/test/generate_testsets.py --dry-run   # chọn trang, KHÔNG gọi LLM
    python src/test/generate_testsets.py             # 25 câu/quyển
    python -m src.test.qa_citation_page --judge      # LLM cứu ca deterministic loại
    python src/test/evaluator.py                     # P/R/MRR + LLM judge 1–5

**Bộ test 12 quyển cũ KHÔNG dùng được nữa** và đã dọn sang
`src/test/testsets/_archive_12books_2026_07/` (ngoài glob): cả hai khoá vàng của nó
đều không khớp metadata chunk hiện hành — `source_book` ghi `"SGK KHTN 6 KNTT.pdf"`
trong khi metadata là `"SGK_KHTN_6_KNTT"`, và `source_page` ghi số trong TÊN FILE
thay vì số trang IN (lệch 1). Bản mới lấy khoá vàng **thẳng từ metadata chunk thật**
nên khớp bởi cấu tạo (D-48).

> ⚠️ **Bộ test do LLM sinh, CHƯA có người duyệt** (`_generation_meta.json` ghi
> `human_reviewed: false`). Mọi báo cáo dùng số từ đây **phải nói rõ điều đó**. Nó
> là thước đo tương đối giữa các cấu hình (ablation), không phải chân lý.

> ⚠️ **`metrics.PAGE_TOLERANCE` = 0.** Chunk không bao giờ vắt qua hai trang, nên
> dung sai ±1 chỉ tính một chunk ở trang KHÁC là "trúng" → thổi recall. Nếu bạn so
> với con số cũ đo bằng ±1 thì đó là hai thước đo khác nhau.

In [ ]:
# Benchmark recall nhanh (base vs rerank, + MRR) — không gọi LLM
!python src/test/recall_at_k.py

## 11. (Tuỳ chọn) Serve API + Cloudflare tunnel để demo

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Backend
!nohup python main.py --api --port 5000 > backend.log 2>&1 &
# Tunnel công khai
!nohup cloudflared tunnel --url http://localhost:5000 > cf_backend.log 2>&1 &

In [ ]:
import time
time.sleep(8)
print('--- backend.log ---');
!tail -n 15 backend.log
print('--- public URL ---')
!grep -o 'https://[^ ]*trycloudflare.com' cf_backend.log | head -n 1

In [ ]:
# Dừng backend + tunnel + giải phóng port
!pkill -f main.py || true
!pkill -f cloudflared || true
!fuser -k 5000/tcp || true

## 12. Sao lưu DB — ĐÃ GỘP vào vòng lặp §9 (D-56 Bước 2/3, DB session-local)

DB nay nằm ở `/content/database` (session, KHÔNG Drive) — zip + tải về xảy ra
SAU MỖI QUYỂN trong vòng lặp §9, không phải một lần ở cuối. Cell dưới đây chỉ
còn dùng khi cần tải lại thủ công (ví dụ notebook bị ngắt giữa vòng lặp).


In [ ]:
import os
src = os.environ["RAG_DATABASE_DIR"]
!zip -r -q /content/database_backup.zip "$src"
from google.colab import files
files.download('/content/database_backup.zip')

## 12. Bake-off OCR — chọn model đọc chữ BẰNG PHÉP ĐO (D-91, D-92, D-93)

**Mục này KHÔNG cần Drive, KHÔNG cần upload gì cả.** Toàn bộ dữ liệu bake-off
(8,3 MB: 97 crop + phiếu người đã duyệt + danh sách ô) nằm **trong repo**, nên
`git clone` ở mục 1 là đã có đủ. Không cần chạy mục 5 (mount Drive), không cần
corpus 4,1 GB.

**Chỉ cần chạy: mục 1 (clone) → mục 12 (cell ngay dưới).** Bỏ qua mục 2–11.

### Đang đo cái gì

Thay Tesseract bằng một model đọc **cả trang**. Nhưng trước khi OCR lại 2 399
trang (bump `TEXT_EXTRACTION_VERSION` = chạy lại toàn bộ), phải biết model nào
thật sự tốt hơn — **trên chính sách này, chấm bởi người**.

**BASELINE Tesseract đã đo** trên gold set 97 ô người duyệt (D-92):

| chỉ số | Tesseract | nghĩa |
|---|---|---|
| **CT** công thức | **0,048** | 4,8% token công thức đọc đúng (45 ô) |
| **DẤU** lỗi dấu | **0,016** | 1,6% từ sai dấu — **mốc RẤT cao phải giữ** |
| **BẢNG** | **0,000** | 0/8, mất hoàn toàn quan hệ hàng/cột |

**Luật chốt:** một engine chỉ THẮNG khi nó **không tệ hơn Tesseract ở cột DẤU**.
93% corpus là chữ thường, nên model giỏi công thức mà sai dấu là model **tệ hơn**.
Và **không model nào trong bốn ứng viên nhắc tới tiếng Việt trong model card** —
đó chính là lý do phải đo chứ không chọn theo danh tiếng (bài học D-47: Vintern-1B
nghe rất hợp lý, chạy thật thì bịa 4/12 crop).

### Chạy thế nào

Cell dưới làm **tất cả trong một lượt**: cài thư viện → chạy engine trên 97 crop
→ in luôn bảng so với Tesseract. Đổi `ENGINE` rồi chạy lại cho engine tiếp theo;
kết quả cũ được giữ nên bảng lớn dần.

**Chạy lần lượt 3 engine** (mỗi lần đổi một dòng, Runtime → Restart giữa các
engine vì thư viện xung khắc nhau):

    nanonets_ocr2_3b  →  mineru25  →  dots_ocr

`paddleocr_vl` là **ô TUỲ CHỌN ở cuối notebook**, không nằm trong đường mặc định:
nó cần `paddlepaddle` cài riêng và bản GPU không có trên PyPI. **Lỗi thì bỏ qua** —
bake-off cần ÍT NHẤT MỘT engine, không cần cả bốn.

Xong, cell tải các file `engine_*.json` về máy để commit.

**Cell sẽ in ra dòng `[nạp] <model> <- transformers.<AutoClass>`.** Đọc dòng đó:
mỗi model dùng một auto-class khác nhau (đo trên `config.json` ngày 26/08 —
Nanonets là `Qwen2_5_VLForConditionalGeneration`, MinerU2.5 là
`Qwen2VLForConditionalGeneration`, dots.ocr khai `AutoModelForCausalLM` qua
`auto_map`), nên script thử theo thứ tự và **nói ra class nào thắng**. Không class
nào nạp được thì nó **raise kèm cả ba lỗi** rồi thoát mã 3 — không có bảng giả.


In [ ]:
# ===== BAKE-OFF OCR — chạy MỘT engine, rồi in bảng so ngay =====
# Cần trước: đã chạy mục 1 (clone repo). KHÔNG cần Drive, KHÔNG cần upload.
# Runtime → Change runtime type → GPU (T4 là đủ).
#
# Chạy theo thứ tự này, mỗi engine một lượt (Runtime → Restart giữa các lượt):
#     mineru25  →  dots_ocr        (nanonets_ocr2_3b đã BỊ LOẠI — xem dưới)
# `paddleocr_vl` để CUỐI CÙNG: nó cần `paddlepaddle` (framework) cài riêng và bản
# GPU không nằm trên PyPI — đã làm hỏng một lượt chạy, xem ô cuối notebook.

# `nanonets_ocr2_3b` ĐÃ BỊ LOẠI 2026-08-26 — vì lý do MÔI TRƯỜNG, không phải
# vì đọc kém: bốn lượt trên T4 đều nạp với `lm_head.weight MISSING` và sinh
# token ngẫu nhiên, mỗi lượt một chuỗi khác nhau dù `do_sample=False`
# (D-99, D-101, D-102). ĐỪNG viết vào báo cáo là 'model này đọc kém tiếng Việt'.
ENGINE = 'mineru25'   # mineru25 | dots_ocr   (nanonets_ocr2_3b: LOẠI, xem trên)

# Cập nhật repo TRƯỚC khi chạy — và `git pull` TRẦN sẽ bị CHẶN từ lượt thứ hai:
# `--compare` ghi đè `bakeoff.csv`, một file SINH RA nhưng nằm trong git, nên
# working tree luôn bẩn sau mỗi lượt. Trả nó về bản gốc rồi mới pull.
# `engine_*.json` là UNTRACKED -> pull không đụng tới, kết quả đã chạy an toàn.
# Runtime -> Restart ĐƯA CWD VỀ `/content`, nên cell này phải tự về repo — nếu
# không sẽ ra `fatal: not a git repository` rồi `No such file or directory`.
# Tự dò thay vì gõ cứng tên thư mục, để clone ở đâu cũng chạy.
import os
if not os.path.exists('scripts/colab_run_ocr_engines.py'):
    for goc in ('/content/project-bio-rag', '/content/drive/MyDrive/project-bio-rag'):
        if os.path.exists(os.path.join(goc, 'scripts/colab_run_ocr_engines.py')):
            os.chdir(goc)
            break
    else:
        raise SystemExit('Không tìm thấy repo — chạy mục 1 (clone) trước.')
print('cwd:', os.getcwd())

!git checkout -- document/review/ocr_gold/bakeoff.csv 2>/dev/null; git pull --ff-only

# GHIM `<5`, KHÔNG PHẢI SỞ THÍCH — ĐO ĐƯỢC (2026-08-26, ba lượt trên T4):
# với transformers **5.15.1**, Nanonets báo `lm_head.weight MISSING` rồi đọc ra
# token ngẫu nhiên, và BA LƯỢT CHO BA CHUỖI KHÁC HẲN NHAU dù `do_sample=False`.
# Greedy + cùng ảnh + cùng prompt mà output đổi giữa các lượt nạp => trọng số
# đổi mỗi lượt => lm_head khởi tạo NGẪU NHIÊN. Checkpoint thì TIED thật
# (0/824 key `lm_head`), `tie_word_embeddings: true` chỉ khai ở `text_config`.
# Cổng `tie_weights()` trong script chạy SAU khi transformers đã dựng model nên
# không nối lại được — phải chặn từ phiên bản thư viện.
!pip -q install -U "transformers>=4.49,<5" accelerate qwen-vl-utils
# MinerU2.5 KHÔNG nhận prompt tự do — nó chạy qua giao diện riêng
# `MinerUClient.two_step_extract()`. Ép prompt vào thì model lặp lại chính câu
# prompt và lộ format nội bộ `<|class_start|>chart<|class_end|>` (đo 2026-08-26).
!pip -q install "mineru-vl-utils[transformers]"
!python -c "import transformers; print('transformers', transformers.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

GOLD = 'document/review/ocr_gold'

# LIMIT = 3 -> chạy thử 3 ô để bắt lỗi thư viện sớm (vài phút, không phải cả giờ).
# LIMIT = 0 -> chạy ĐỦ 97 ô. Đo được 2026-08-26: ~29,5 s/ô -> ~48 phút/engine,
#              cộng ~150 s nạp model.
# Đổi MỘT số này rồi chạy lại cell — cố ý không dùng hai khối comment, vì bỏ
# comment khối thứ hai làm cell nạp model HAI lần (~5 phút vứt đi mỗi engine).
# Script tự nối tiếp phần đã chạy, nên 3 ô của lượt thử không phải chạy lại.
LIMIT = 3

# ĐÃ CHẠY TRƯỚC BẢN VÁ CỔNG TIE (2026-08-26)? Bỏ comment dòng dưới, chạy MỘT
# lần: script NỐI TIẾP phần đã có, nên mấy ô rác cũ sẽ được giữ nguyên và đi
# thẳng vào bảng. Chạy xong rồi thì comment lại — nếu không, mỗi lần đứt phiên
# Colab bạn mất cả lượt 48 phút thay vì chạy tiếp.
# !rm -f document/review/ocr_gold/engine_$ENGINE.json

# `&&` là CỐ Ý: engine lỗi thì DỪNG, không chạy tiếp sang --compare. Không có nó,
# một lượt nạp model thất bại vẫn in ra bảng baseline trông như kết quả bình
# thường — đúng bệnh D-83.
!python scripts/colab_run_ocr_engines.py --engine $ENGINE \
    --crops-dir $GOLD/crops --out-dir $GOLD --limit $LIMIT \
  && python -m src.test.ocr_bakeoff --compare

# VỚI `LIMIT = 3` BẢNG CHƯA PHẢI KẾT QUẢ: nó in `—  —  —  CHƯA ĐỦ: thiếu 94/97 ô`
# cho engine, và đó là ĐÚNG. Trước bản vá D-96 nó in `0.000 0.000 0.000`, mà
# DẤU 0,000 là điểm HOÀN HẢO ở đúng cột quyết định thắng/thua (một từ mất hẳn
# không phải "lỗi dấu"), nên bảng đó nói NGƯỢC sự thật chứ không chỉ nói thiếu.
#
# `lm_head.weight | MISSING` KHÔNG vô hại — dòng cũ ở đây nói ngược và đã sai.
# ĐO ĐƯỢC (2026-08-26): với transformers 5.15.1, Nanonets báo MISSING rồi đọc
# 3/3 ô ra TOKEN NGẪU NHIÊN đa ngôn ngữ. `model.safetensors.index.json` không có
# key `lm_head.weight` nào trong 824 key (checkpoint TIED) và
# `tie_word_embeddings: true` chỉ khai trong `text_config`, top-level là `None`
# -> transformers 5.x không buộc trọng số -> lm_head NGẪU NHIÊN -> rác.
# Script nay có CỔNG: tự gọi `tie_weights()` và in `[vá] …`; không sửa được thì
# RAISE. Thấy `[vá] xong:` là đã ổn.


# ĐỌC Ô BẰNG MẮT trước khi tin bất kỳ con số nào (CẤM #11). In cạnh nhau bản
# NGƯỜI · engine · tesseract. Chạy được ngay với lượt thử 3 ô — nếu chữ engine
# là rác hoặc engine BỊA chữ không có trên ảnh thì LOẠI THẲNG dù CT cao (D-47),
# và không phải đốt 48 phút cho engine đó.
!python -m src.test.ocr_bakeoff --doi-chieu $ENGINE --so-o 10
# --loai cong_thuc | doi_chung | bang | so   để lọc theo loại ô


In [ ]:
# Xong CẢ BỐN engine thì tải kết quả về máy để commit vào repo.
# (Chỉ vài chục KB — đây là phép đo, phải đi cùng lịch sử của nó.)
from google.colab import files
import glob

for f in sorted(glob.glob('document/review/ocr_gold/engine_*.json')):
    print('tải:', f)
    files.download(f)
files.download('document/review/ocr_gold/bakeoff.csv')


In [ ]:
# ===== TÙY CHỌN: paddleocr_vl — chỉ chạy khi ba engine kia đã xong =====
# Nó cần `paddlepaddle` (framework) cài RIÊNG, `pip install paddleocr` KHÔNG kéo
# theo. Bản GPU không nằm trên PyPI (PyPI chỉ có paddlepaddle-gpu 2.6.2 quá cũ
# cho paddleocr 3.x) nên phải lấy từ index riêng của Paddle.
#
# Nếu cell này lỗi: BỎ QUA paddleocr_vl. Ba engine kia đã đủ để chọn.
# Đừng đốt thời gian ở đây — bake-off cần ÍT NHẤT MỘT engine, không cần cả bốn.

!python -m pip install -q paddlepaddle-gpu==3.0.0 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip -q install "paddleocr[doc-parser]"
!python -c "import paddle; paddle.utils.run_check()" | tail -2

GOLD = 'document/review/ocr_gold'
!python scripts/colab_run_ocr_engines.py --engine paddleocr_vl \
    --crops-dir $GOLD/crops --out-dir $GOLD --limit 3 \
  && python -m src.test.ocr_bakeoff --compare
